# ReAct Agent Walkthrough

This is the core notebook for understanding the system as an agent rather than just a collection of tools.

The main question here is not simply whether the final estimate is good. The main question is whether the sequence of reasoning steps makes sense:

- what information did the agent notice first
- which tool did it decide to call next
- how did the observation change its confidence
- when did it decide it had enough evidence to stop


## Step 1: Load A Small Set Of Campaigns And Build The Agent

This cell loads a few campaigns from the saved dataset and constructs the LangGraph ReAct agent.

Why this matters:

- using only a few campaigns keeps the walkthrough readable
- loading from disk means you can inspect the exact same campaign row in the CSV if you want more context
- the agent object bundles the prompt, parsing logic, graph loop, and tool routing in one place


In [ ]:
from iroas_agent.agent import default_agent
from iroas_agent.data import DEFAULT_DATASET_PATH, load_campaign_dataset

In [ ]:
campaigns = load_campaign_dataset(DEFAULT_DATASET_PATH)[:3]
agent = default_agent(model_name='gpt-4o-mini', max_steps=5)
campaigns[0].observed.to_agent_dict()

## Step 2: Run The Agent On One Campaign

This cell executes the full LangGraph loop for a single campaign.

What is happening under the hood:

- the prompt file is loaded from the repository
- the model emits a Thought, an Action, and an Action Input
- if the action is a tool call, the tool runs and returns an Observation
- the graph loops until the model chooses `finish`

Why this matters:

- it makes the agent behavior inspectable step by step
- it enforces the project rule that the agent must rely on tools rather than directly computing iROAS


In [ ]:
result = agent.run_campaign(campaigns[0])
result['prediction']

## Step 3: Read The Full Trajectory

This is the most important inspection cell in the notebook.

As you read the output, ask:

- did the agent prefer stronger causal evidence when available
- did it use diagnostics to reason about reliability rather than blindly trusting the first estimator it saw
- if it compared tools, did the comparison actually help clarify uncertainty
- did it stop at a sensible point rather than making unnecessary calls


In [ ]:
for step in result['trajectory']:
    print('Thought:', step['thought'])
    print('Action:', step['action'])
    print('Action Input:', step['action_input'])
    print('Observation:', step['observation'])
    print('-' * 80)

## Step 4: Compare Prediction To Hidden Truth

This final cell compares the agent's answer with the hidden evaluation target.

Why this matters:

- it reminds us that a plausible reasoning trace is not enough on its own
- we still want to know whether the agent landed near the true iROAS
- later notebooks will focus more directly on failure modes when these two diverge


In [ ]:
{
    'predicted_iROAS': result['prediction']['estimated_iROAS'],
    'true_iROAS': campaigns[0].hidden.true_iroas,
    'final_estimator_used': result['prediction']['final_estimator_used'],
}

## What To Take Away

A useful agent walkthrough should leave you able to explain not only the final answer, but the path the model took to get there.

That path is the main object of study in this project.